# `tea_api` walkthrough — running the TEA cost model from Python

This notebook is a guided tour of `tea/tea_api.py`, the programmatic Python
API for the TEA (techno-economic analysis) manufacturing cost model. It
exists so a researcher can call the cost model from their own script or
notebook — for example to screen many candidate alloy compositions — without
running the web GUI's server and without going through `tea_script.py`'s
interactive `input()` prompts.

**What this notebook covers, in order:**

1. Running the model for a single, existing material (by name).
2. Inspecting the full result: cost breakdown, per-element material cost, warnings.
3. Defining a brand-new, custom material composition on the fly.
4. Blending two materials into one effective material.
5. Defining a brand-new, custom process (instead of using one from the database).
6. Batch-screening a *list* of candidate materials with `tea_api.screen()`.
7. (Optional) flattening batch results into a `pandas` DataFrame.
8. Batch-screening *every material file in a directory* with `tea_api.screen_directory()`.
9. A one-page cheat sheet of every public function.

**Setup note:** see the *Setup* section right below - `pip install -e .` once from `tea/`.


## Setup

`tea/` is a small, pip-installable package (`tea/pyproject.toml`) - the
core cost model (`tea_api`, `tea1`, `material_def`, `process_def`,
`component_def`, `cost_variables`) has no third-party dependencies. This
only needs to be done once per environment:

```bash
cd tea
pip install -e .
```

`-e` ("editable") installs it in place, pointing back at this checkout on
disk rather than copying files.

If you're using a conda environment as this notebook's
kernel, activate it first (`conda activate env`) before running
`pip install -e .`, then select `env` as the kernel in
Jupyter/VS Code. (Need `pandas` for the optional DataFrame cell in Part 5b?
`pip install -e ".[notebook]"` instead pulls that in too.)


In [1]:
import tea_api


## Part 1 — Running a single, existing material

`tea_api.evaluate(material, processes, volume_mm3, production_qty, ...)` is
the core single-material entry point. Here `material` is just the name of a
material that's already in `tea/materials_database/` (or a directory you
point at with `material_dir=`), and `processes` is a list of process names
already in `tea/processes_database/`.

It returns a plain dict — nothing web/GUI-specific, no chart images, just
numbers and strings you can use however you like:

```python
{
    "summary": {...},               # from Component.manufacturing_cost()
    "material_composition": [...],  # per-element cost breakdown
    "warnings": [...],              # e.g. missing cost-database entries
}
```

In [2]:
result = tea_api.evaluate(
    "EROFER97",                    # material name, looked up in materials_database/
    ["CNC", "Hot Rolling"],        # process names, looked up in processes_database/
    volume_mm3=3_000_000,          # part volume, mm^3
    production_qty=100,            # units to be produced
)

summary = result["summary"]
print(f"Total cost:  ${summary['Total cost']:.2f}")
print(f"Unit cost:   ${summary['Unit cost']:.2f} / kg")
print(f"Mass:        {summary['Mass']:.3f} kg")

# summary["Processes"] has one entry per process: Pc is the base processing
# cost curve value at this quantity, Rc is the relative cost multiplier
# (Cmp * Cc * Cs * max(Ct, Cf)), and Cost = Rc * Pc.
for p in summary["Processes"]:
    print(f"  {p['Process']:<15} Pc=${p['Pc']:.2f}  Rc={p['Rc']:.2f}  Cost=${p['Cost']:.2f}")

Total cost:  $210.62
Unit cost:   $8.78 / kg
Mass:        24.000 kg
  CNC             Pc=$18.48  Rc=4.00  Cost=$73.93
  Hot Rolling     Pc=$30.58  Rc=2.00  Cost=$61.17


## Part 1a — Selecting a geometry

Passing `geometry=` to `evaluate()` (a name string looked up in
`tea/geometries_database/`, or an already-built `Geometry` object) is all it
takes to have `Cc`/`Cs`/`Ct`/`Cf` come from that geometry's own maps instead
of the `1.0` default - see Part 1c below for the full resolution rules.
Here's the same `EROFER97` run as Part 1, but pointed at the
`"Plasma Facing Surface"` geometry already in `tea/geometries_database/`:


In [3]:
result_geo = tea_api.evaluate(
    "EROFER97",
    ["CNC", "Hot Rolling"],
    volume_mm3=3_000_000,
    production_qty=100,
    geometry="Plasma Facing Surface",  # name string, looked up in geometries_database/
)

for p in result_geo["summary"]["Processes"]:
    print(f"  {p['Process']:<15} Rc={p['Rc']:.3f}  Cost=${p['Cost']:.2f}")

# Same Rc as Part 1's baseline (CNC=4.0, Hot Rolling=2.0): "Plasma Facing
# Surface" defines Cc=Cs=Ct=Cf=1.0 for every process it covers, by design -
# it's meant as the DFM-ideal reference geometry, not a worst-case one. Swap
# in a different geometry name to see Rc actually move.


  CNC             Rc=4.000  Cost=$73.93
  Hot Rolling     Rc=2.000  Cost=$61.17


## Part 1b — The material cost breakdown and warnings

`result["material_composition"]` is a list of rows, one per element in the
material, each with its weight percent, whether it's an `"alloy"` addition
or a `"residual"` (present but not intentionally/cost-bearing added — e.g.
trace carbon or sulfur), its absolute cost contribution in `$/kg`, and its
`fraction` of the material's total `$/kg` cost.

`result["warnings"]` surfaces non-fatal issues — most commonly, an element in
the composition that has no entry in the cost database (its contribution is
silently treated as \$0/kg, so the warning is your only signal that the
total material cost may be an underestimate).

In [4]:
for row in result["material_composition"]:
    print(
        f"  {row['element']:<3} {row['type']:<9} "
        f"wt={row['wt']:>6.2f}%  ${row['dollar_per_kg']:.4f}/kg  "
        f"({row['fraction']*100:.1f}% of material cost)"
    )

print("\nWarnings:", result["warnings"])

  Fe  rem       wt= 88.85%  $1.0662/kg  (33.9% of material cost)
  Cr  alloy     wt=  9.00%  $0.9900/kg  (31.5% of material cost)
  W   alloy     wt=  1.10%  $0.5500/kg  (17.5% of material cost)
  Ta  alloy     wt=  0.12%  $0.4800/kg  (15.3% of material cost)
  V   alloy     wt=  0.20%  $0.0420/kg  (1.3% of material cost)
  B   alloy     wt=  0.00%  $0.0110/kg  (0.3% of material cost)
  Mn  alloy     wt=  0.40%  $0.0076/kg  (0.2% of material cost)
  C   residual  wt=  0.11%  $0.0000/kg  (0.0% of material cost)
  O   residual  wt=  0.01%  $0.0000/kg  (0.0% of material cost)
  N   residual  wt=  0.03%  $0.0000/kg  (0.0% of material cost)
  P   residual  wt=  0.01%  $0.0000/kg  (0.0% of material cost)
  S   residual  wt=  0.01%  $0.0000/kg  (0.0% of material cost)
  Nb  residual  wt=  0.01%  $0.0000/kg  (0.0% of material cost)
  Mo  residual  wt=  0.01%  $0.0000/kg  (0.0% of material cost)
  Ni  residual  wt=  0.01%  $0.0000/kg  (0.0% of material cost)
  Cu  residual  wt=  0.01%  $0.0000/

## Part 1c — Default cost coefficients, and where each one comes from

Notice Part 1 never mentioned a geometry, yet `summary["Processes"]` still had
an `Rc` value for each process. Every process's cost is
`Cost = Rc * Pc`, where:

```
Rc = Cmp * Cc * Cs * max(Ct, Cf)
```

Each factor comes from a different place:

* **`Cmp`** (material/process compatibility) is tied to the **material**.
  For a database material like `EROFER97`, it comes from that material's own
  `Cmp_map`, looked up per process. A from-scratch/custom material (Part 2
  below) has no `Cmp_map` at all, so `Cmp` must be supplied as a per-process
  `"Cmp"` override instead - `evaluate()` raises `TeaError` if it's missing
  and there's nothing to fall back on.
* **`Cc` / `Cs` / `Ct` / `Cf`** (complexity / size / tolerance / surface-finish)
  are tied to the **geometry**. **When no `geometry=` argument is passed, all
  four default to `1.0`** (the DFM-ideal baseline) for every process. You can
  raise any of them above 1.0 yourself, geometry or not, with a per-process
  override in that process's spec dict, e.g. `{"name": "CNC", "Cc": 1.3}`.
* **`Wc`** (scrap/waste) is tied to **neither** material nor geometry - there's
  no `Wc_map` anywhere. It always defaults to `1.0`; the only way to change it
  is a per-process `"Wc"` override.

**Passing `geometry=`** (a name string looked up in
`tea/geometries_database/`, e.g. `geometry="Plasma Facing Surface"`) supplies
`Cc`/`Cs`/`Ct`/`Cf` automatically, **per process**, from that geometry's own
`Cc_map`/`Cs_map`/`Ct_map`/`Cf_map`. For any process the geometry's map
covers, its value **wins over any per-process override you supplied for that
coefficient**. For a process the geometry *doesn't* cover, resolution falls
back to your override (or the `1.0` default) exactly as if no geometry had
been given at all. `Cmp` and `Wc` are untouched by geometry either way.

So Part 1's `EROFER97` example above used `Cc=Cs=Ct=Cf=Wc=1.0` for both
processes (no geometry, no overrides), which is why `CNC`'s `Rc` (4.0) and
`Hot Rolling`'s `Rc` (2.0) exactly equal EROFER97's own `Cmp_map` values for
those processes - the other coefficients were all 1 and dropped out.


Here's Part 1's example again with **no `geometry=` at all** - `Cc`/`Cs`/`Ct`/`Cf`
supplied directly as per-process overrides instead, exactly the mechanism
described above:


In [5]:
result_override = tea_api.evaluate(
    "EROFER97",
    [{"name": "CNC", "Cc": 1.3, "Cs": 1.1, "Ct": 1.2, "Cf": 1.05}],  # no geometry= at all
    volume_mm3=3_000_000,
    production_qty=100,
)

for p in result_override["summary"]["Processes"]:
    print(f"  {p['Process']:<15} Rc={p['Rc']:.3f}  Cost=${p['Cost']:.2f}")

# No geometry passed - Cc/Cs/Ct/Cf come entirely from the per-process
# overrides above (Cmp still comes from EROFER97's own Cmp_map, unaffected):
# Rc = Cmp(4) * Cc(1.3) * Cs(1.1) * max(Ct(1.2), Cf(1.05)) = 6.864


  CNC             Rc=6.864  Cost=$126.87


## Part 2 — A brand-new, custom material composition

This is the interesting case for alloy screening: you don't have to look a
material up by name, you can hand `evaluate()` a composition directly. The
`material` argument is shape-sniffed — pass a `dict` with a `"composition"`
key and it's treated as a custom material, built on the fly via
`material_def.build_material()`.

The composition dict maps element symbol -> `{"wt": weight_percent, "type":
"alloy" | "residual"}`. You don't need to list every element: whatever's
left over after the listed elements is assigned to `remainder_element`
automatically (so it always sums to 100 wt%).

**One thing to know:** a from-scratch material has no compatibility-cost
(`Cmp`) data for any process — that only exists for materials already
characterized in the database. So when evaluating a custom material, give
each process spec a `"Cmp"` override (its cost relative to processing an
ideal material with this process) instead of relying on the material to
supply one.

In [6]:
v4cr4ti = {
    "name": "V-4Cr-4Ti",
    "density": 6.06,  # g/cm^3
    "composition": {
        "Cr": {"wt": 4.0, "type": "alloy"},
        "Ti": {"wt": 4.0, "type": "alloy"},
        # everything not listed here falls into the remainder element below
    },
    "remainder_element": "V",
}

result = tea_api.evaluate(
    v4cr4ti,
    processes=[{"name": "CNC", "Cmp": 1.0}],  # Cmp override - see note above
    volume_mm3=1_000_000,
    production_qty=50,
)

print(f"Custom material total cost: ${result['summary']['Total cost']:.2f}")

Custom material total cost: $142.86


## Part 3 — Blending two materials

`blend_materials()` models a part made from two materials as one
homogeneous effective material (rule-of-mixtures density, mass-weighted
composition and cost) — useful for e.g. a functionally-graded armor/
structure combination, modeled as a single bulk material rather than a
spatially resolved gradient.

Pass a dict with `material_a`/`material_b`/`fraction_a`/`basis` instead of a
`"composition"` key. `material_a`/`material_b` can each be a name string
**or** another custom composition dict — they're resolved the same way
`evaluate()`'s own `material` argument is, recursively, so you can blend two
not-yet-in-the-database candidates together too.

In [7]:
blend = {
    "name": "Eurofer/V4Cr4Ti Blend",
    "material_a": "EROFER97",        # a name string looked up in the database...
    "material_b": v4cr4ti,           # ...or a custom composition dict, mixed freely
    "fraction_a": 0.5,               # EROFER97's share of the blend
    "basis": "volume",               # "volume" or "mass"
}

result = tea_api.evaluate(
    blend,
    [{"name": "CNC", "Cmp": 1.0}],  # a blend also starts with an empty Cmp_map
    volume_mm3=1_000_000,
    production_qty=50,
)

print(f"Blend total cost: ${result['summary']['Total cost']:.2f}")

Blend total cost: $93.77


## Part 4 — A brand-new, custom process

The same shape-sniffing works for `processes`: a dict with
`tooling_level`/`equipment_level`/`time_level` (one of `process_def.COST_LEVELS`
each) builds a brand-new process via `process_def.build_process()`, computing
its cost coefficients (`alphaT`, `beta`) from those categorical levels rather
than looking an existing process up by name.

Like a custom material, a custom process isn't in any material's `Cmp_map`
yet, so it also needs a `"Cmp"` override in its spec.

In [8]:
custom_process = {
    "name": "Laser Cladding",
    "tooling_level": "Medium",
    "equipment_level": "High",
    "time_level": "Low",
    "description": "Experimental additive repair process",
    "Cmp": 1.2,
}

result = tea_api.evaluate("EROFER97", [custom_process], volume_mm3=1_000_000, production_qty=50)
print(f"Custom process total cost: ${result['summary']['Total cost']:.2f}")

Custom process total cost: $97.38


## Part 5 — Batch screening a list of candidates with `screen()`

This is the main event for alloy screening: `tea_api.screen(candidates,
processes, volume_mm3, production_qty, ...)` runs `evaluate()` once per
candidate, sharing the same processes/geometry/volume/production quantity
across all of them. Each entry in `candidates` can be anything `evaluate()`'s
`material` argument accepts — a name string, a custom composition dict, a
blend dict, or an already-built `Material` object.

**Important: invalid candidates don't abort the batch.** If one composition
is malformed (bad density, weights that don't sum sensibly, etc.), `screen()`
catches it, records the error, and moves on to the next candidate — exactly
what you want when screening dozens or hundreds of hypothetical alloys and
expecting some fraction of them to be invalid.

Below, the last candidate has a deliberately invalid (negative) density to
show this in action.

In [9]:
candidates = [
    "EROFER97",  # you can mix an existing database material in with custom ones
    {"name": "V1", "density": 7.6, "composition": {"Cr": {"wt": 9.0, "type": "alloy"}}, "remainder_element": "Fe"},
    {"name": "V2", "density": 7.7, "composition": {"Cr": {"wt": 12.0, "type": "alloy"}}, "remainder_element": "Fe"},
    {"name": "Bad idea", "density": -1, "composition": {"Cr": {"wt": 9.0, "type": "alloy"}}, "remainder_element": "Fe"},
]

results = tea_api.screen(
    candidates,
    processes=[{"name": "CNC", "Cmp": 1.0}],
    volume_mm3=1_000_000,
    production_qty=50,
)

# Each result is {"material": <label>, "ok": bool, "result": <evaluate() output or None>, "error": <str or None>}
for row in results:
    if row["ok"]:
        print(f"{row['material']:<12} OK      Total cost = ${row['result']['summary']['Total cost']:.2f}")
    else:
        print(f"{row['material']:<12} FAILED  {row['error']}")

EROFER97     OK      Total cost = $103.11
V1           OK      Total cost = $35.31
V2           OK      Total cost = $37.78
Bad idea     FAILED  Error loading material, 'Bad idea'. Material skipped. Density must be greater than zero.


## Part 5b — (Optional) flattening results into a `pandas` DataFrame

`screen()`'s output is plain nested dicts on purpose — no `tea_api`
dependency on `pandas`. But since the results are just dicts, flattening
them into a DataFrame for further analysis/plotting/filtering is a couple of
lines if you want it.

In [10]:
import pandas as pd

rows = []
for r in results:
    if r["ok"]:
        rows.append({
            "material": r["material"],
            "total_cost": r["result"]["summary"]["Total cost"],
            "unit_cost": r["result"]["summary"]["Unit cost"],
            "error": None,
        })
    else:
        rows.append({"material": r["material"], "total_cost": None, "unit_cost": None, "error": r["error"]})

df = pd.DataFrame(rows)
df

,material,total_cost,unit_cost,error
0,EROFER97,103.107444,12.888431,None
1,V1,35.306509,4.645593,None
2,V2,37.778509,4.906300,None
3,Bad idea,NaN,NaN,"Error loading material, 'Bad idea'. Material s..."


## Part 6 — Screening every material file in a directory with `screen_directory()`

If your candidate alloys already exist as individual `.py` files — the same
format as `tea/materials_database/*.py`, each defining module-level
`name`, `density`, `composition`, `remainder_element`, and (optionally)
`Cmp_map` — you don't need to build a Python list of dicts by hand.
`tea_api.screen_directory(material_dir, processes, volume_mm3,
production_qty, ...)` loads every `.py` file in the directory and screens
all of them, with the same skip-and-record behavior as `screen()`.

This is also how the web GUI's "Run All Materials in Directory & Export CSV"
button works under the hood — same function, same behavior, just called from
`server.py` instead of a notebook.

To keep this notebook self-contained (no external files required to run
it), the cell below builds a temporary directory with three material files
— two valid, one deliberately invalid — then cleans it up afterward.

In [11]:
import tempfile, os, textwrap, shutil

candidate_dir = tempfile.mkdtemp(prefix="tea_candidates_")

# Each file below is exactly what a materials_database/*.py file looks like:
# module-level name/density/composition/remainder_element/Cmp_map variables.
candidate_files = {
    "v1.py": '''
        name = "V1"
        density = 7.6
        composition = {"Cr": {"wt": 9.0, "type": "alloy"}}
        remainder_element = "Fe"
        Cmp_map = {"CNC": 1.0}
    ''',
    "v2.py": '''
        name = "V2"
        density = 7.7
        composition = {"Cr": {"wt": 12.0, "type": "alloy"}}
        remainder_element = "Fe"
        Cmp_map = {"CNC": 1.0}
    ''',
    "bad.py": '''
        name = "Bad"
        density = -1  # deliberately invalid, to show skip-and-record below
        composition = {"Cr": {"wt": 9.0, "type": "alloy"}}
        remainder_element = "Fe"
        Cmp_map = {}
    ''',
}
for filename, content in candidate_files.items():
    with open(os.path.join(candidate_dir, filename), "w") as f:
        f.write(textwrap.dedent(content).strip() + "\n")

print("Candidate directory:", candidate_dir)
print(sorted(os.listdir(candidate_dir)))

Candidate directory: /tmp/tea_candidates_1574dkij
['bad.py', 'v1.py', 'v2.py']


In [12]:
dir_results = tea_api.screen_directory(
    candidate_dir,
    processes=["CNC"],  # an existing process name this time - each file already supplies its own Cmp_map
    volume_mm3=1_000_000,
    production_qty=50,
)

for row in dir_results:
    status = "OK" if row["ok"] else "FAILED"
    print(f"{row['material']:<12} {status:<7} {row['error'] or ''}")

Bad          FAILED  Error loading material, 'Bad'. Material skipped. Density must be greater than zero.
V1           OK      
V2           OK      


In [13]:
# Clean up the temporary directory now that we're done with it.
shutil.rmtree(candidate_dir)
print("cleaned up:", not os.path.exists(candidate_dir))

cleaned up: True


## Cheat sheet

| Function | Purpose |
|---|---|
| `tea_api.evaluate(material, processes, volume_mm3, production_qty, *, geometry=None, component_name="Component", material_dir=None, process_dir=None, geometry_dir=None)` | Run the cost model for **one** material. Returns `{"summary", "material_composition", "warnings"}`. |
| `tea_api.screen(candidates, processes, volume_mm3, production_qty, *, ...)` | Run the cost model for a **list** of candidate materials, sharing the same processes/geometry/quantity. Skips and records failures instead of aborting. |
| `tea_api.screen_directory(material_dir, processes, volume_mm3, production_qty, *, ...)` | Same as `screen()`, but candidates come from every `.py` file in a directory (same format as `materials_database/`). |
| `tea_api.resolve_material(material, material_dir=None)` | Lower-level: resolve a name/dict/`Material` spec into a `Material` object, without running a cost calculation. |
| `tea_api.resolve_process(process, process_dir=None)` / `resolve_processes(list)` | Lower-level: resolve a process spec into a `Process` object (or list of them). |
| `tea_api.resolve_geometry(geometry, geometry_dir=None)` | Lower-level: resolve a geometry name/object into a `Geometry` object. |
| `tea_api.build_component(name, volume_mm3, production_qty, material, processes, process_specs=None, geometry=None)` | Lower-level: build a `Component` from already-resolved objects, e.g. if you want to call `component.fabrication_cost_curve()` yourself for a custom sensitivity sweep. |
| `tea_api.TeaError` | Raised for any invalid input (failed material/process build, missing `Cmp`, etc.). |
| `tea_api.TeaNotFoundError` | A subclass of `TeaError`, raised specifically when a named material/process/geometry isn't found. |

**Geometry:** every function above accepts an optional `geometry=` keyword
(a name string looked up in `tea/geometries_database/`, or a `Geometry`
object) plus a matching `geometry_dir=` for a custom directory — it supplies
per-process `Cc`/`Cs`/`Ct`/`Cf` coefficients automatically wherever the
geometry covers that process, the same way it works in the web GUI.